In [1]:
# Cell 1: Imports and config
import os, json, numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, ParameterGrid

SEED = 42
np.random.seed(SEED)
OUT_DIR = "results/baselines"
os.makedirs(OUT_DIR, exist_ok=True)


In [2]:
# Data
X = np.random.randn(200, 2)
y = ((X[:, 0] * X[:, 1]) > 0).astype(int)


In [3]:
# CV and grid
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = {"C": [0.1, 1.0, 10.0], "gamma": ["scale", 0.5, 1.0]}


In [4]:
# CV loop
results = []
best_acc, best_cfg = -np.inf, None

for cfg in ParameterGrid(grid):
    accs, aucs = [], []
    for tr, te in cv.split(X, y):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"], random_state=SEED))
        ])
        pipe.fit(X_tr, y_tr)
        y_pred = pipe.predict(X_te)

        accs.append(accuracy_score(y_te, y_pred))
        try:
            aucs.append(roc_auc_score(y_te, y_pred))
        except Exception:
            aucs.append(np.nan)

    mean_acc = float(np.nanmean(accs))
    mean_auc = float(np.nanmean(aucs))
    results.append({"cfg": cfg, "acc": {"mean": mean_acc}, "auc": {"mean": mean_auc}})
    if mean_acc > best_acc:
        best_acc, best_cfg = mean_acc, cfg

with open(os.path.join(OUT_DIR, "cv_results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(OUT_DIR, "best_cfg.json"), "w") as f:
    json.dump({"best_acc": best_acc, "best_cfg": best_cfg}, f, indent=2)

print("Best:", best_cfg, "Acc:", best_acc)


Best: {'C': 10.0, 'gamma': 'scale'} Acc: 0.9349999999999999
